# IOAI — 2025 Stage 1 Coin Counting Machine (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
if not os.path.exists('data/train.pkl'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-1-coin-counting-machine/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 동전 세기 기계 — 객체 검출 (Coin Counting / 베이스라인)

폴란드 AI 올림피아드 II · 2025 · 1단계. 사진 속 **폴란드 동전**을 모두 찾아(경계상자) 액면가로
분류하는 **객체 검출(object detection)** 문제. 9개 클래스(1gr·2gr·5gr·10gr·20gr·50gr·1zł·2zł·5zł).

**지표**: **mAP**(mean Average Precision, IoU 0.5:0.95) — 검출의 표준 지표. 점수 = `clip(mAP,0.2,0.85)→0~100`.

이 노트북은 **베이스라인** = 원문제가 제시한 예시(슬라이딩 윈도우): resnet18 분류기를 64×64 크롭으로
학습(9동전+배경 10클래스) → 윈도우를 stride 32로 훑어 배경 아닌 곳을 검출. **val mAP ≈ 0.16 → 0점**.
모범답안(사전학습 Faster R-CNN 미세조정, mAP≈0.91→100점)을 참고해 `YourDetector` 를 개선하라.

**제출**: `submission.csv` — `image_id,x1,y1,x2,y2,label,score` (예측 박스 1개당 1행).


In [ ]:
# 데이터 준비 (Colab: 자동 다운로드 / DGX: data/ 이미 존재)
import os, urllib.request, zipfile
if not os.path.exists("data/train.pkl"):
    url = "https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-1-coin-counting-machine/data.zip"
    urllib.request.urlretrieve(url, "d.zip"); zipfile.ZipFile("d.zip").extractall("data")

import pickle, csv, numpy as np, torch, torch.nn as nn
import torchvision.transforms.v2 as T
dev = "cuda" if torch.cuda.is_available() else "cpu"
_tf = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

def load_train(f="data/train.pkl"):
    data = pickle.load(open(f, "rb")); out = []
    for s in data:
        out.append((_tf(s["image"]),
                    torch.as_tensor(np.array(s["boxes"]), dtype=torch.float32).reshape(-1,4),
                    torch.as_tensor(np.array(s["labels"]), dtype=torch.long).reshape(-1)))
    return out

def load_val_images(f="data/val_images.pkl"):
    return [_tf(s["image"]) for s in pickle.load(open(f, "rb"))]   # 라벨 없음(정답 held-out)

train = load_train(); val_imgs = load_val_images()
print("train", len(train), "val", len(val_imgs), "| dev", dev)


In [ ]:
from torchvision.models import resnet18
from torchvision.ops import box_iou

# 64x64 크롭 분류기 학습용 전처리 (90% 동전 주변 / 10% 배경)
class Prep:
    def __init__(self, n=128, box=64): self.n=n; self.box=box
    def _lbl(self, cb, boxes, labels):
        for b, l in zip(boxes, labels):
            if box_iou(torch.tensor(cb).view(1,4).float(), b.view(1,4)) > 0.5: return int(l)
        return 9   # 배경
    def __call__(self, batch):
        crops, labels = [], []
        for img, boxes, labs in batch:
            for _ in range(self.n):
                if torch.rand(1) < 0.1:
                    cx = torch.randint(0, img.shape[2], (1,)).item(); cy = torch.randint(0, img.shape[1], (1,)).item()
                else:
                    j = torch.randint(0, labs.shape[0], (1,)).item()
                    cx = int((boxes[j,0]+boxes[j,2])//2) + torch.randint(-10,10,(1,)).item()
                    cy = int((boxes[j,1]+boxes[j,3])//2) + torch.randint(-10,10,(1,)).item()
                x1 = int(np.clip(cx-self.box//2, 0, img.shape[2]-self.box)); y1 = int(np.clip(cy-self.box//2, 0, img.shape[1]-self.box))
                crops.append(img[:, y1:y1+self.box, x1:x1+self.box]); labels.append(self._lbl((x1,y1,x1+self.box,y1+self.box), boxes, labs))
        return torch.stack(crops), torch.tensor(labels)

class YourDetector(nn.Module):
    """베이스라인: resnet18 크롭분류기 + 슬라이딩 윈도우(stride 32). 배경(9) 아닌 윈도우를 검출로."""
    def __init__(self, crop=64, stride=32):
        super().__init__(); self.clf = resnet18(weights=None, num_classes=10); self.cs=crop; self.st=stride
    @torch.no_grad()
    def forward(self, image):
        image = image.to(dev); crops=[]; pos=[]
        for y in range(0, image.shape[1]-self.cs, self.st):
            for x in range(0, image.shape[2]-self.cs, self.st):
                crops.append(image[:, y:y+self.cs, x:x+self.cs]); pos.append((x,y))
        p = self.clf(torch.stack(crops)); prob = torch.softmax(p, dim=1); found=[]
        for k,(x,y) in enumerate(pos):
            lab = int(p[k].argmax())
            if lab != 9: found.append((x, y, x+self.cs, y+self.cs, lab, float(prob[k,lab])))
        return found

det = YourDetector().to(dev); prep = Prep(128, 64)
opt = torch.optim.Adam(det.clf.parameters(), lr=1e-3); crit = nn.CrossEntropyLoss()
dl = torch.utils.data.DataLoader(train, batch_size=2, shuffle=True, collate_fn=prep)
det.clf.train()
for ep in range(30):
    for X, y in dl:
        opt.zero_grad(); loss = crit(det.clf(X.to(dev)), y.to(dev)); loss.backward(); opt.step()
print("분류기 학습 완료")
det.clf.eval()


In [ ]:
# val 이미지 예측 -> submission.csv
rows = []
with torch.no_grad():
    for img_id, img in enumerate(val_imgs):
        for (x1, y1, x2, y2, label, score) in det(img):
            rows.append([img_id, round(float(x1),2), round(float(y1),2), round(float(x2),2), round(float(y2),2), int(label), round(float(score),5)])
with open("submission.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["image_id","x1","y1","x2","y2","label","score"]); w.writerows(rows)
print("submission.csv 저장:", len(rows), "박스,", len(val_imgs), "이미지")


### 다음 단계
슬라이딩 윈도우는 느리고 중복검출이 많아 mAP 가 낮다(≈0.16). `YourDetector` 를 **사전학습 검출기**
(Faster R-CNN 등) 미세조정으로 바꾸면 mAP 가 ~0.9 로 뛴다. 모범답안 참고.


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)